# 02 - Entrenamiento y selección del mejor modelo

Notebook de semana 3 para fine-tuning de modelos de detección sobre CarDD COCO con tres variantes de Faster R-CNN y evaluación basada en mAP.

## 1. Imports y configuración

In [1]:
from pathlib import Path
import json
import random
import shutil
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from prod.detection_dataset import (
    CarDamageDetectionDataset,
    ComposeDetection,
    RandomHorizontalFlipDetection,
    ToTensorDetection,
    collate_fn,
)
from prod.detection_metrics import evaluate_map
from prod.detection_models import (
    build_optimizer,
    build_scheduler,
    create_model_from_config,
    describe_parameter_counts,
)
from prod.detection_training import run_detection_experiment, load_checkpoint


In [2]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "dev" / "experiments"
FINAL_MODEL_PATH = PROJECT_ROOT / "dev" / "modelo.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("DATA_DIR:", DATA_DIR)


Device: cpu
DATA_DIR: /home/joaquin/code/proyecto_final_redes/data


## 2. Verificación del dataset local

Este notebook requiere las imágenes reales del dataset en `train2017`, `val2017` y `test2017`.

In [3]:
required_dirs = [
    DATA_DIR / "CarDD_release" / "CarDD_COCO" / "train2017",
    DATA_DIR / "CarDD_release" / "CarDD_COCO" / "val2017",
    DATA_DIR / "CarDD_release" / "CarDD_COCO" / "test2017",
]
missing_dirs = [path for path in required_dirs if not path.exists()]
if missing_dirs:
    raise FileNotFoundError(
        "Faltan las carpetas de imagenes necesarias para entrenar:\n" + "\n".join(str(path) for path in missing_dirs)
    )

print("Dataset listo para entrenamiento.")


Dataset listo para entrenamiento.


## 3. Datasets y DataLoaders

In [4]:
BATCH_SIZE = 2
NUM_WORKERS = 0
BASE_MODEL_NAME = "fasterrcnn"

train_transform = ComposeDetection([
    ToTensorDetection(),
    RandomHorizontalFlipDetection(p=0.5),
])

eval_transform = ComposeDetection([
    ToTensorDetection(),
])

train_dataset = CarDamageDetectionDataset(
    data_dir=DATA_DIR,
    split="train",
    transform=train_transform,
    model_name=BASE_MODEL_NAME,
    resize=False,
    image_size=None,
)

val_dataset = CarDamageDetectionDataset(
    data_dir=DATA_DIR,
    split="val",
    transform=eval_transform,
    model_name=BASE_MODEL_NAME,
    resize=False,
    image_size=None,
)

test_dataset = CarDamageDetectionDataset(
    data_dir=DATA_DIR,
    split="test",
    transform=eval_transform,
    model_name=BASE_MODEL_NAME,
    resize=False,
    image_size=None,
)

NUM_CLASSES = len(train_dataset.class_to_idx)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

print("Clases:", NUM_CLASSES)
print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))


Clases: 7
Train samples: 2816
Val samples: 810
Test samples: 374


## 4. Tres variantes de Faster R-CNN

Se comparan tres configuraciones: solo cabeza, backbone parcial y backbone completo con resize fijo.

In [5]:
EXPERIMENTS = [
    {
        "name": "fasterrcnn_head_only",
        "model_name": "fasterrcnn",
        "weights": "DEFAULT",
        "num_classes": NUM_CLASSES,
        "trainable_backbone_layers": 0,
        "optimizer_name": "sgd",
        "lr": 0.005,
        "momentum": 0.9,
        "weight_decay": 0.0005,
        "scheduler_name": "step",
        "step_size": 3,
        "gamma": 0.1,
        "num_epochs": 8,
        "resize": False,
        "image_size": None,
    },
    {
        "name": "fasterrcnn_partial_backbone",
        "model_name": "fasterrcnn",
        "weights": "DEFAULT",
        "num_classes": NUM_CLASSES,
        "trainable_backbone_layers": 2,
        "optimizer_name": "sgd",
        "lr": 0.003,
        "momentum": 0.9,
        "weight_decay": 0.0005,
        "scheduler_name": "step",
        "step_size": 3,
        "gamma": 0.1,
        "num_epochs": 8,
        "resize": False,
        "image_size": None,
    },
    {
        "name": "fasterrcnn_full_backbone_fixed_size",
        "model_name": "fasterrcnn",
        "weights": "DEFAULT",
        "num_classes": NUM_CLASSES,
        "trainable_backbone_layers": 5,
        "optimizer_name": "sgd",
        "lr": 0.001,
        "momentum": 0.9,
        "weight_decay": 0.0005,
        "scheduler_name": "step",
        "step_size": 3,
        "gamma": 0.1,
        "num_epochs": 8,
        "resize": True,
        "image_size": (640, 640),
    },
]

pd.DataFrame(EXPERIMENTS)[[
    "name",
    "trainable_backbone_layers",
    "lr",
    "num_epochs",
    "resize",
    "image_size",
]]


,name,trainable_backbone_layers,lr,num_epochs,resize,image_size
0,fasterrcnn_head_only,0,0.005,8,False,None
1,fasterrcnn_partial_backbone,2,0.003,8,False,None
2,fasterrcnn_full_backbone_fixed_size,5,0.001,8,True,"(640, 640)"


## 5. Helpers de entrenamiento

In [6]:
def build_dataloaders_for_experiment(experiment):
    train_dataset = CarDamageDetectionDataset(
        data_dir=DATA_DIR,
        split="train",
        transform=train_transform,
        model_name=experiment["model_name"],
        resize=experiment["resize"],
        image_size=experiment["image_size"],
    )
    val_dataset = CarDamageDetectionDataset(
        data_dir=DATA_DIR,
        split="val",
        transform=eval_transform,
        model_name=experiment["model_name"],
        resize=experiment["resize"],
        image_size=experiment["image_size"],
    )
    test_dataset = CarDamageDetectionDataset(
        data_dir=DATA_DIR,
        split="test",
        transform=eval_transform,
        model_name=experiment["model_name"],
        resize=experiment["resize"],
        image_size=experiment["image_size"],
    )

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
    )
    test_loader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
    )

    return train_loader, val_loader, test_loader


def plot_history(history, title):
    history_df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    axes[0].set_title(f"{title} - Loss")
    axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["map"], label="mAP@50:95")
    axes[1].plot(history_df["epoch"], history_df["map_50"], label="mAP@50")
    axes[1].set_title(f"{title} - mAP")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## 6. Ejecución de experimentos

In [7]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
experiment_summaries = []
experiment_runs = {}

for experiment in EXPERIMENTS:
    print(f"\n===== {experiment['name']} =====")

    train_loader_exp, val_loader_exp, test_loader_exp = build_dataloaders_for_experiment(experiment)
    model = create_model_from_config(experiment)
    optimizer = build_optimizer(
        model,
        optimizer_name=experiment["optimizer_name"],
        lr=experiment["lr"],
        momentum=experiment["momentum"],
        weight_decay=experiment["weight_decay"],
    )
    scheduler = build_scheduler(
        optimizer,
        scheduler_name=experiment["scheduler_name"],
        step_size=experiment["step_size"],
        gamma=experiment["gamma"],
    )

    parameter_counts = describe_parameter_counts(model)
    print(parameter_counts)

    run_result = run_detection_experiment(
        model=model,
        train_loader=train_loader_exp,
        val_loader=val_loader_exp,
        optimizer=optimizer,
        device=DEVICE,
        num_epochs=experiment["num_epochs"],
        experiment_name=experiment["name"],
        config=experiment,
        scheduler=scheduler,
        output_dir=ARTIFACTS_DIR,
        class_metrics=True,
    )

    experiment_runs[experiment["name"]] = {
        "config": experiment,
        "run_result": run_result,
        "test_loader": test_loader_exp,
    }

    history_df = pd.DataFrame(run_result["history"])
    best_row = history_df.loc[history_df["map"].idxmax()].to_dict()
    experiment_summaries.append({
        "name": experiment["name"],
        "trainable_backbone_layers": experiment["trainable_backbone_layers"],
        "resize": experiment["resize"],
        "image_size": experiment["image_size"],
        "best_epoch": run_result["best_epoch"],
        "best_map": best_row.get("map"),
        "best_map_50": best_row.get("map_50"),
        "best_val_loss": best_row.get("val_loss"),
        "checkpoint_path": run_result["best_checkpoint_path"],
    })

    plot_history(run_result["history"], experiment["name"])



===== fasterrcnn_head_only =====
{'trainable_parameters': 17869874, 'frozen_parameters': 23454912, 'total_parameters': 41324786}
[fasterrcnn_head_only] Starting training for 8 epoch(s)


Train Epoch 1:   0%|          | 0/1408 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 7. Comparación de resultados

Las métricas principales para comparar son `mAP@50:95` y `mAP@50`.

In [ ]:
results_df = pd.DataFrame(experiment_summaries).sort_values(by="best_map", ascending=False)
results_df


## 8. Selección del mejor modelo y evaluación final en test

In [ ]:
best_experiment_name = results_df.iloc[0]["name"]
best_experiment = experiment_runs[best_experiment_name]
best_config = best_experiment["config"]
best_checkpoint_path = best_experiment["run_result"]["best_checkpoint_path"]

best_model = create_model_from_config(best_config).to(DEVICE)
_ = load_checkpoint(best_model, best_checkpoint_path, device=DEVICE)
test_metrics = evaluate_map(
    model=best_model,
    dataloader=best_experiment["test_loader"],
    device=DEVICE,
    class_metrics=True,
)

print(json.dumps({
    "best_experiment": best_experiment_name,
    "test_map": test_metrics.get("map"),
    "test_map_50": test_metrics.get("map_50"),
    "test_map_75": test_metrics.get("map_75"),
    "test_mar_100": test_metrics.get("mar_100"),
}, indent=2))


## 9. Guardado del modelo final

In [ ]:
shutil.copy2(best_checkpoint_path, FINAL_MODEL_PATH)
print("Modelo final guardado en:", FINAL_MODEL_PATH)
